# Messages in AutoGen

In the [previous notebook](01_first_agent.ipynb) we built our first agent and called `agent.run(task="...")`. Behind the scenes, that string was wrapped into a **message** and the agent's reply came back as more messages.

This notebook is a tour of the **message system** in AutoGen v0.4 — the data structures every agent uses to talk.

## What we'll cover
1. What a message looks like
2. `TextMessage` — the workhorse
3. `MultiModalMessage` — text + images

> **Mental model:** a message is a typed record with a `source` (who sent it), a `content` (what they said), and a `type` (what kind of message). Agents pass these around; the UI renders them.

## 1. Setup
Same as last time — load the API key and create a model client.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found. Add it to .env"

from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
print("Ready.")

## 2. What a message looks like

Every message you'll send or receive in AutoGen is a small Python object — a Pydantic model — with the same three core fields:

- **`source`** — who sent it (an agent's `name`, or `"user"`)
- **`content`** — what they said (text, or a mix of text and images)
- **`type`** — the kind of message (e.g. `"TextMessage"`)

The two kinds we'll meet here are `TextMessage` (plain text) and `MultiModalMessage` (text + images). Let's import them:

In [ ]:
from autogen_agentchat.messages import TextMessage, MultiModalMessage

print("Imported:", TextMessage.__name__, "and", MultiModalMessage.__name__)

## 3. `TextMessage` — the workhorse

99% of agent communication is `TextMessage`. You rarely need to construct one by hand — `run(task="hi")` does it for you — but knowing the shape makes the rest of the API click.

In [ ]:
msg = TextMessage(content="Hello, agent!", source="user")

print(f"type:    {msg.type}")
print(f"source:  {msg.source}")
print(f"content: {msg.content}")

# Messages are Pydantic models — they serialize to JSON cleanly:
print("\nAs JSON:")
print(msg.model_dump_json(indent=2))

## 4. `MultiModalMessage` — text + images

Vision-capable models (like `gpt-4o` and `gpt-4o-mini`) can read images. AutoGen represents this with `MultiModalMessage`, whose `content` is a **list** that mixes strings and `Image` objects.

You can hand a `MultiModalMessage` to `run(task=...)` directly — it accepts any message, not just a string.

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_core import Image
from PIL import Image as PILImage
import io, urllib.request

# Lorem Picsum serves stable, permissive images for placeholders.
# id=237 is a small photo of a puppy — handy for a predictable demo.
url = "https://picsum.photos/id/237/240/160"
raw = urllib.request.urlopen(url).read()
pil_img = PILImage.open(io.BytesIO(raw))
img = Image(pil_img)

vision_agent = AssistantAgent(
    name="viewer",
    model_client=model_client,
    system_message="Describe images in one short sentence.",
)

mm = MultiModalMessage(
    content=["What's in this image?", img],
    source="user",
)

result = await vision_agent.run(task=mm)
print(result.messages[-1].content)

## Recap

- A message is a small typed record with three core fields: `source`, `content`, `type`.
- `TextMessage` is what you'll use 99% of the time — plain text in `content`.
- `MultiModalMessage` puts a list of strings and `Image` objects in `content`, and you can hand it to `run(task=...)` directly.

### Try it yourself
1. Build a `TextMessage` by hand, print its JSON, then change one field and pass it to `run(task=...)`.
2. Send a `MultiModalMessage` containing two images and ask the agent to compare them.
3. Swap the system message on `vision_agent` and rerun — see how the description style changes.